In [1]:
import pandas as pd
import numpy as np
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import pickle
import warnings
warnings.filterwarnings('ignore')

In [4]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"

respuesta = urllib.request.urlopen(url)
contenido = respuesta.read().decode('utf-8', errors='replace')

# Procesar línea a línea para evitar errores de comillas
filas = []
for linea in contenido.strip().split('\n'):
    # Separar por la ÚLTIMA coma (la polarity siempre es el último campo)
    ultima_coma = linea.rfind(',')
    if ultima_coma == -1:
        continue
    texto = linea[:ultima_coma]
    polarity = linea[ultima_coma+1:].strip()
    
    # Separar package_name del review por la primera coma
    primera_coma = texto.find(',')
    if primera_coma == -1:
        continue
    package = texto[:primera_coma].strip()
    review = texto[primera_coma+1:].strip().strip('"')
    
    filas.append([package, review, polarity])

# Crear DataFrame 
df = pd.DataFrame(filas[1:], columns=['package_name', 'review', 'polarity'])

# Convertir polarity a número
df['polarity'] = pd.to_numeric(df['polarity'], errors='coerce')

print(f"   OK: Datos cargados")
print(f"   - Total de filas: {df.shape[0]}")
print(f"   - Columnas: {list(df.columns)}")

if df.shape[0] < 800:
    print(f"\n   ADVERTENCIA: Solo se cargaron {df.shape[0]} filas")
    print(f"   Se esperaban ~890 filas")
    print(f"   Verifica la conexion a internet o la URL")
else:
    print(f"   OK: Dataset completo cargado")

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# Eliminar nulos
filas_antes = len(df)
df = df.dropna()
print(f"   - Filas despues de limpiar nulos: {df.shape[0]}")
if filas_antes - len(df) > 0:
    print(f"   - Se eliminaron {filas_antes - len(df)} filas con nulos")


   OK: Datos cargados
   - Total de filas: 891
   - Columnas: ['package_name', 'review', 'polarity']
   OK: Dataset completo cargado
   - Filas despues de limpiar nulos: 891


# EXPLORACION DE DATOS

In [5]:
print("\nDistribucion de clases:")
print(df['polarity'].value_counts())

neg = (df['polarity']==0).sum()
pos = (df['polarity']==1).sum()
print(f"\n   - Negativas (0): {neg} ({neg/len(df)*100:.1f}%)")
print(f"   - Positivas (1): {pos} ({pos/len(df)*100:.1f}%)")

# Estadisticas de texto
print(f"\nEstadisticas de las resenas:")
df['review_length'] = df['review'].astype(str).str.len()
print(f"   - Longitud promedio: {df['review_length'].mean():.0f} caracteres")
print(f"   - Longitud minima: {df['review_length'].min()}")
print(f"   - Longitud maxima: {df['review_length'].max()}")


Distribucion de clases:
polarity
0    584
1    307
Name: count, dtype: int64

   - Negativas (0): 584 (65.5%)
   - Positivas (1): 307 (34.5%)

Estadisticas de las resenas:
   - Longitud promedio: 231 caracteres
   - Longitud minima: 4
   - Longitud maxima: 910


# PREPROCESAMIENTO

In [6]:
# Eliminar columna package_name 
if 'package_name' in df.columns:
    df = df.drop('package_name', axis=1)

# Limpiar texto
df['review'] = df['review'].astype(str).str.strip().str.lower()

# Eliminar textos muy cortos 
df = df[df['review'].str.len() > 10]
print(f"   - Textos procesados: {df.shape[0]}")

# Eliminar columna temporal
if 'review_length' in df.columns:
    df = df.drop('review_length', axis=1)

print(f"\n   Ejemplo de texto procesado:")
print(f"   '{df['review'].iloc[0][:80]}...'")

   - Textos procesados: 884

   Ejemplo de texto procesado:
   'privacy at least put some option appear offline. i mean for some people like me ...'


# DIVIDIR DATOS

In [7]:
X = df['review']
y = df['polarity']

print(f"   - Total de ejemplos: {len(X)}")
print(f"   - Distribucion: {y.value_counts().to_dict()}")

# Dividir: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Mantener proporcion de clases
)

print(f"\n   - Train: {len(X_train)} ejemplos")
print(f"   - Test: {len(X_test)} ejemplos")
print(f"   - Distribucion train: {y_train.value_counts().to_dict()}")
print(f"   - Distribucion test: {y_test.value_counts().to_dict()}")

   - Total de ejemplos: 884
   - Distribucion: {0: 584, 1: 300}

   - Train: 707 ejemplos
   - Test: 177 ejemplos
   - Distribucion train: {0: 467, 1: 240}
   - Distribucion test: {0: 117, 1: 60}


# VECTORIZACION

In [8]:
# Crear vectorizador
vec = CountVectorizer(
    stop_words='english',
    max_features=1000,  # Aumentado porque tenemos mas datos
    min_df=2,           # Palabra debe aparecer al menos 2 veces
    max_df=0.8          # Ignorar palabras muy frecuentes
)

# Transformar
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

print(f"\n   OK: Vocabulario: {len(vec.vocabulary_)} palabras")
print(f"   OK: Matriz train: {X_train_vec.shape}")
print(f"   OK: Matriz test: {X_test_vec.shape}")

# Mostrar palabras de ejemplo
palabras = sorted(list(vec.vocabulary_.keys()))[:15]
print(f"\n   Primeras 15 palabras del vocabulario:")
print(f"   {', '.join(palabras)}")


   OK: Vocabulario: 1000 palabras
   OK: Matriz train: (707, 1000)
   OK: Matriz test: (177, 1000)

   Primeras 15 palabras del vocabulario:
   10, 100, 12, 15, 2015, 3g, 4g, 50, ability, able, access, account, accounts, actually, ad


# ENTRENAR LOS 3 NAIVE BAYES

In [9]:
resultados = {}

# Modelo 1: Gaussian Naive Bayes
print("\n   A) GAUSSIAN NAIVE BAYES")
print("      (Asume distribucion normal)")
gnb = GaussianNB()
gnb.fit(X_train_vec.toarray(), y_train) 
y_pred_gnb = gnb.predict(X_test_vec.toarray())
acc_gnb = accuracy_score(y_test, y_pred_gnb)
resultados['Gaussian'] = acc_gnb
print(f"      Precision: {acc_gnb:.4f} ({acc_gnb*100:.2f}%)")

# Modelo 2: Multinomial Naive Bayes
print("\n   B) MULTINOMIAL NAIVE BAYES")
print("      (Cuenta frecuencias - MEJOR PARA TEXTO)")
mnb = MultinomialNB()
mnb.fit(X_train_vec, y_train)
y_pred_mnb = mnb.predict(X_test_vec)
acc_mnb = accuracy_score(y_test, y_pred_mnb)
resultados['Multinomial'] = acc_mnb
print(f"      Precision: {acc_mnb:.4f} ({acc_mnb*100:.2f}%)")

# Modelo 3: Bernoulli Naive Bayes
print("\n   C) BERNOULLI NAIVE BAYES")
print("      (Presencia/ausencia de palabras)")
bnb = BernoulliNB()
bnb.fit(X_train_vec, y_train)
y_pred_bnb = bnb.predict(X_test_vec)
acc_bnb = accuracy_score(y_test, y_pred_bnb)
resultados['Bernoulli'] = acc_bnb
print(f"      Precision: {acc_bnb:.4f} ({acc_bnb*100:.2f}%)")


   A) GAUSSIAN NAIVE BAYES
      (Asume distribucion normal)
      Precision: 0.7401 (74.01%)

   B) MULTINOMIAL NAIVE BAYES
      (Cuenta frecuencias - MEJOR PARA TEXTO)
      Precision: 0.8305 (83.05%)

   C) BERNOULLI NAIVE BAYES
      (Presencia/ausencia de palabras)
      Precision: 0.8418 (84.18%)


# COMPARAR Y ELEGIR MEJOR

In [11]:
for modelo, prec in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    print(f"   {modelo:15s}: {prec:.4f} ({prec*100:.2f}%)")

mejor = max(resultados, key=resultados.get)
print(f"\n   MEJOR MODELO: {mejor} Naive Bayes")

# Seleccionar modelo
if mejor == 'Multinomial':
    modelo_nb = mnb
    y_pred_nb = y_pred_mnb
elif mejor == 'Bernoulli':
    modelo_nb = bnb
    y_pred_nb = y_pred_bnb
else:
    modelo_nb = gnb
    y_pred_nb = y_pred_gnb

   Bernoulli      : 0.8418 (84.18%)
   Multinomial    : 0.8305 (83.05%)
   Gaussian       : 0.7401 (74.01%)

   MEJOR MODELO: Bernoulli Naive Bayes


In [13]:
# Matriz de confusion
print("\nMatriz de Confusion:")
cm = confusion_matrix(y_test, y_pred_nb)
print(cm)

# Metricas
aciertos = accuracy_score(y_test, y_pred_nb) * len(y_test)
errores = len(y_test) - aciertos
print(f"\n   Aciertos: {int(aciertos)}/{len(y_test)}")
print(f"   Errores: {int(errores)}/{len(y_test)}")

# Reporte completo
print("Reporte de Clasificacion:")
print(classification_report(y_test, y_pred_nb, target_names=['Negativo', 'Positivo']))



Matriz de Confusion:
[[106  11]
 [ 17  43]]

   Aciertos: 149/177
   Errores: 28/177
Reporte de Clasificacion:
              precision    recall  f1-score   support

    Negativo       0.86      0.91      0.88       117
    Positivo       0.80      0.72      0.75        60

    accuracy                           0.84       177
   macro avg       0.83      0.81      0.82       177
weighted avg       0.84      0.84      0.84       177



# OPTIMIZAR CON RANDOM FOREST

In [14]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1  # Usar todos los nucleos
)

rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"\n   Random Forest: {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print(f"   Naive Bayes:   {resultados[mejor]:.4f} ({resultados[mejor]*100:.2f}%)")

if acc_rf > resultados[mejor]:
    print(f"\n   -> Random Forest MEJORO el resultado!")
    mejora = (acc_rf - resultados[mejor]) * 100
    print(f"   -> Mejora: +{mejora:.2f}%")
    modelo_final = rf
    acc_final = acc_rf
    nombre_final = "Random Forest"
else:
    print(f"\n   -> Naive Bayes sigue siendo mejor")
    modelo_final = modelo_nb
    acc_final = resultados[mejor]
    nombre_final = f"{mejor} NB"


   Random Forest: 0.7910 (79.10%)
   Naive Bayes:   0.8418 (84.18%)

   -> Naive Bayes sigue siendo mejor


# GUARDAR MODELO

In [15]:
with open('modelo_sentimientos.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)
    
with open('vectorizador.pkl', 'wb') as f:
    pickle.dump(vec, f)

print(f"   OK: Modelo guardado ({nombre_final})")
print("   OK: Archivos creados:")
print("       - modelo_sentimientos.pkl")
print("       - vectorizador.pkl")

   OK: Modelo guardado (Bernoulli NB)
   OK: Archivos creados:
       - modelo_sentimientos.pkl
       - vectorizador.pkl
